**器材借用系統**

In [9]:
!pip install -q groq gradio

In [10]:
import sqlite3
from google.colab import userdata #important
from groq import Groq
import gradio as gr

In [11]:
# acces API
client =Groq(api_key = userdata.get('GROQ_API_KEY'))
print("Groq連線成功!")

Groq連線成功!


In [12]:
#database: delete,create,insert
#1. connecting to sqlite3
conn = sqlite3.connect('equipment.db')
cursor = conn.cursor()

#2. delete
cursor.execute('''DROP TABLE IF EXISTS items''')
cursor.execute('''DROP TABLE IF EXISTS borrow_logs''')

#3.create table:items
cursor.execute('''
  CREATE TABLE items(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT,
  status TEXT)
  ''')


#4. create items table
cursor.execute('''
  CREATE TABLE borrow_logs (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  item_name TEXT,
  borrow_item TIMESTAMPE DEFAULT CURRENT_TIMESTAMP)
  ''')

#5. insert item into items:在資料表items中新增器材
initial_data=[('麥克風','可借用'),('筆電','可借用'),('HDMI','已借出'),('3軸穩定器','已借出')]
cursor.executemany("INSERT INTO items(name,status) VALUES(?,?)",initial_data)

#6. close database
conn.commit()
conn.close()
print("資料庫建立完成")

資料庫建立完成


In [13]:
#buliding fuction with GenAI

def borrow_item(user_input):


  prompt = f'''
  系辦目前只有4種器材:[麥克風,筆電,HDMI,3軸穩定器],
  請從學生的話裡面，找出他想要借用的器材，
  規則:只回傳器材名稱(例如:麥克風、筆電、HDMI、3軸穩定器),
  如果學生講的不是這4種，請回傳"未知"。絕對不要有多餘的字。

  學生的話:{user_input}
  '''

  response= client.chat.completions.create(
      model='openai/gpt-oss-120b',
      messages=[{'role':'system','content':prompt}],
      temperature=0.0 #請AI完整照指令
  )

  return response.choices[0].message.content.strip()

In [14]:
#borrow system
def borrow_system(user_say):
  #1.先請AI找出要借用的設備名稱
  target_item = borrow_item(user_say)
  if target_item not in ['麥克風','筆電','HDMI','3軸穩定器']:
    return f'抱歉，系統找不到{target_item}這個設備'

  #2.如果找到，顯示狀態
  conn =sqlite3.connect('equipment.db')
  cursor = conn.cursor()
  cursor.execute("SELECT status FROM items WHERE name =?",(target_item,))
  row=cursor.fetchone()

  current_status = row[0]

  print(f'設備"{target_item}"的狀態是"{current_status}"')

  #3.按照狀態給出不同回應
  if current_status == '已借出':
    conn.close()
    return f'抱歉,{target_item}已被借出。'
  else:
    cursor.execute('UPDATE items SET status="已借出" WHERE name =?',(target_item,))
    cursor.execute('INSERT INTO borrow_logs (item_name) VALUES (?)',(target_item,))
    conn.close()
    return f'已將{target_item}借出。'

In [15]:
#testing
borrow_system('我要借用教室，需要麥克風')

設備"麥克風"的狀態是"可借用"


'已將麥克風借出。'

In [16]:
#用gradio呈現網頁
def web_intergerface(message):
  if not message.strip():
    return "請輸入想借用的設備"
  reply = borrow_system(message)
  return reply

demo = gr.Interface(
     fn=web_intergerface,
     inputs=gr.Textbox(label='輸入您的器材需求'),
     outputs=gr.Textbox(label='系統回應'),
     title='簡易器材借用系統',
     description='Groq API+sqlite3=系統'
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0063efa13e53a59119.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
